# Tokenizer Script-Routing — 최종 코퍼스 10,020건 문자권 분기 검증 노트북

문자 범위 기반 분기 규칙을 최종 코퍼스 10,020건에 적용하고, 원본 토크나이저 실행 결과(`wordcloud_by_language_v7.json`)와 절대 수치로 대조한다.

- 원본(`wordcloud_by_language_v7.json`)의 버킷 집계는 **한 불릿이 여러 버킷에 들어갈 수 있는 다중 라벨**("해당 버킷 토큰이 1개 이상 있는 불릿 수")이다.
- 이 노트북은 ① 단일 버킷 배정과 ② 다중 라벨 재현 두 가지를 모두 계산한다.

In [1]:
import json
import math
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.max_colwidth", 60)
pd.set_option("display.width", 140)


def find_repo_root():
    for p in [Path.cwd(), *Path.cwd().parents]:
        if (p / "data" / "v7_final" / "fandoms_v3_100.json").exists():
            return p
    raise FileNotFoundError("저장소 루트(data/v7_final/fandoms_v3_100.json)를 찾지 못함 — 저장소 안에서 실행하세요")


REPO = find_repo_root()
DATA_DIR = REPO / "data" / "v7_final"                       # 최종 산출물(10,020건 라이브 + 동결 스냅샷 7,350건)
ROUNDS_DIR = REPO / "data" / "v7_rounds"                    # 병합 로그 r1~r72


def load_json(path):
    with open(path, encoding="utf-8") as f:
        return json.load(f)


def flatten_bullets(fandoms):
    rows = []
    for rec in fandoms:
        for kind in ("loyalty", "spillover"):
            for item in rec.get(kind, []):
                rows.append({"fandom": rec["fandom"], "category": rec.get("category"),
                             "bullet_type": kind, "text": item.get("t", "") or "", "url": item.get("u", "") or ""})
    return pd.DataFrame(rows)

import re
fandoms = load_json(DATA_DIR / "fandoms_v3_100.json")
wc = load_json(DATA_DIR / "wordcloud_by_language_v7.json")
bullets_df = flatten_bullets(fandoms)
print(f"평탄화된 불릿 수: {len(bullets_df)} | wordcloud JSON total_bullets: {wc['total_bullets']} | total_tokens: {wc['total_tokens']}")
print("원본 버킷:", [(b["bucket"], b["n_bullets_with_any_token"]) for b in wc["buckets"]])

평탄화된 불릿 수: 10020 | wordcloud JSON total_bullets: 10020 | total_tokens: 170725
원본 버킷: [('한국어', 8942), ('영어', 6425), ('비영어(스페인어·프랑스어·포르투갈어·인도네시아어·말레이어·필리핀어·튀르키예어 등)', 219), ('러시아어(키릴문자)', 45), ('베트남어(라틴 확장 성조 부호)', 30), ('일본어', 155), ('중국어', 398), ('태국어', 56)]


## 1. 문자 범위 기반 스크립트 분기 — 단일 버킷 배정

In [2]:
RE_HANGUL = re.compile(r"[가-힣]")
RE_KANA = re.compile(r"[぀-ゟ゠-ヿ]")
RE_HANZI = re.compile(r"[一-鿿]")
RE_THAI = re.compile(r"[฀-๿]")
RE_CYRILLIC = re.compile(r"[Ѐ-ӿ]")
RE_VIET = re.compile(r"[đơưĐƠƯ]|[Ḁ-ỿ]")
RE_LATIN_EXT = re.compile(r"[À-ɏ]")
RE_ASCII_LATIN = re.compile(r"[A-Za-z]")

BK_KO, BK_EN, BK_OTHER, BK_RU, BK_VI, BK_JA, BK_ZH, BK_TH = [b["bucket"] for b in wc["buckets"]]

def classify_single(text):
    if RE_KANA.search(text): return BK_JA
    if RE_HANZI.search(text): return BK_ZH
    if RE_THAI.search(text): return BK_TH
    if RE_HANGUL.search(text): return BK_KO
    if RE_CYRILLIC.search(text): return BK_RU
    if RE_VIET.search(text): return BK_VI
    if RE_LATIN_EXT.search(text): return BK_OTHER
    if RE_ASCII_LATIN.search(text): return BK_EN
    return "기타/미분류"

bullets_df["script_bucket"] = bullets_df["text"].apply(classify_single)
bullets_df["script_bucket"].value_counts()

script_bucket
한국어                                              8385
영어                                                902
중국어                                               439
일본어                                               159
태국어                                                56
비영어(스페인어·프랑스어·포르투갈어·인도네시아어·말레이어·필리핀어·튀르키예어 등)      39
베트남어(라틴 확장 성조 부호)                                  21
러시아어(키릴문자)                                         19
Name: count, dtype: int64

## 2. 다중 라벨 재현 — 원본 `tokenize()` 분기(가나→ja, 한자(가나 없음)→zh, 태국문자→th) + 일반 경로 토큰의 사후 재분류

In [3]:
WORD_RE = re.compile(r"[^\s\.,;:!?\"'“”‘’()\[\]{}<>·•\-–—/\\|~`^*+=_@#$%&《》「」『』]+")

RE_CJK_RUN = re.compile(r"[一-鿿]{2,}")   # jieba 토큰은 2자 이상만 남긴다고 보고 1자 한자만 있는 불릿은 중국어 토큰 없음으로 처리

def buckets_of(text):
    hit = set()
    if RE_KANA.search(text): hit.add(BK_JA)
    elif RE_CJK_RUN.search(text): hit.add(BK_ZH)
    if RE_THAI.search(text): hit.add(BK_TH)
    for tok in WORD_RE.findall(text):
        if re.fullmatch(r"[\d,\.%]+", tok): continue
        if RE_HANGUL.search(tok): hit.add(BK_KO)
        elif RE_CYRILLIC.search(tok): hit.add(BK_RU)
        elif RE_VIET.search(tok): hit.add(BK_VI)
        elif RE_KANA.search(tok) or RE_HANZI.search(tok) or RE_THAI.search(tok): continue   # 형태소 경로가 처리
        elif tok.isascii():
            if RE_ASCII_LATIN.search(tok): hit.add(BK_EN)
        else: hit.add(BK_OTHER)   # 악센트 라틴·×·기타 비ASCII 기호 토큰 → 비영어(원본 top_30에 '2×'가 있는 것과 같은 규칙)
    return hit

bullets_df["buckets_multi"] = bullets_df["text"].apply(buckets_of)
multi_counts = {b["bucket"]: int(bullets_df["buckets_multi"].apply(lambda s: b["bucket"] in s).sum()) for b in wc["buckets"]}
single_counts = bullets_df["script_bucket"].value_counts().to_dict()
comparison = pd.DataFrame([{"문자권": b["bucket"], "원본(다중라벨)": b["n_bullets_with_any_token"], "원본 비중": b["bullet_share"],
                            "재현(다중라벨)": multi_counts[b["bucket"]], "차이": multi_counts[b["bucket"]] - b["n_bullets_with_any_token"],
                            "단일버킷 배정": single_counts.get(b["bucket"], 0)} for b in wc["buckets"]])
print("다중 라벨 재현 — 절대오차 합:", int(comparison["차이"].abs().sum()), "| 정확 일치 버킷:", int((comparison["차이"] == 0).sum()), "/ 8")
print("(태국어·러시아어·베트남어는 정확 일치. 일본어·중국어는 분기 조건은 같지만 원본이 형태소 결과에서 불용어·1자 토큰을 걸러 토큰 0개가 된 불릿만큼 +3~4건 차이,")
print(" 영어·비영어·한국어는 원본 불용어 목록·토큰 정규식이 저장소에 없어 그 차이만큼 근사 — 순위·비중 구조는 동일)")
comparison

다중 라벨 재현 — 절대오차 합: 187 | 정확 일치 버킷: 3 / 8
(태국어·러시아어·베트남어는 정확 일치. 일본어·중국어는 분기 조건은 같지만 원본이 형태소 결과에서 불용어·1자 토큰을 걸러 토큰 0개가 된 불릿만큼 +3~4건 차이,
 영어·비영어·한국어는 원본 불용어 목록·토큰 정규식이 저장소에 없어 그 차이만큼 근사 — 순위·비중 구조는 동일)


,문자권,원본(다중라벨),원본 비중,재현(다중라벨),차이,단일버킷 배정
0,한국어,8942,0.8924,8946,4,8385
1,영어,6425,0.6412,6271,-154,902
2,비영어(스페인어·프랑스어·포르투갈어·인도네시아어·말레이어·필리핀어·튀르키예어 등),219,0.0219,241,22,39
3,러시아어(키릴문자),45,0.0045,45,0,19
4,베트남어(라틴 확장 성조 부호),30,0.0030,30,0,21
5,일본어,155,0.0155,159,4,159
6,중국어,398,0.0397,401,3,439
7,태국어,56,0.0056,56,0,56


## 3. 소수 언어권 순위 — 원본 vs 재현

In [4]:
minor = [BK_JA, BK_ZH, BK_TH, BK_RU, BK_VI]
mc = comparison[comparison["문자권"].isin(minor)].copy()
mc["원본 순위"] = mc["원본(다중라벨)"].rank(ascending=False).astype(int); mc["재현 순위"] = mc["재현(다중라벨)"].rank(ascending=False).astype(int)
print(f"소수 언어권 5개 중 순위 동일: {(mc['원본 순위'] == mc['재현 순위']).sum()} / 5")
mc[["문자권", "원본(다중라벨)", "원본 순위", "재현(다중라벨)", "재현 순위"]]

소수 언어권 5개 중 순위 동일: 5 / 5


,문자권,원본(다중라벨),원본 순위,재현(다중라벨),재현 순위
3,러시아어(키릴문자),45,4,45,4
4,베트남어(라틴 확장 성조 부호),30,5,30,5
5,일본어,155,2,159,2
6,중국어,398,1,401,1
7,태국어,56,3,56,3


## 4. 실제 불릿 예시 — 문자권별 1건씩 + 원본 상위 단어

In [5]:
top_words = {b["bucket"]: [w["word"] for w in b["top_30"][:8]] for b in wc["buckets"]}
for b in wc["buckets"]:
    sample = bullets_df[bullets_df["buckets_multi"].apply(lambda s: b["bucket"] in s)]
    if len(sample) == 0:
        print(f"[{b['bucket']}] 없음\n"); continue
    row = sample.iloc[0]
    print(f"[{b['bucket']}] 원본 상위어: {'·'.join(top_words[b['bucket']])}")
    print(f"   예시({row['fandom']}): {row['text'][:90]}{'…' if len(row['text']) > 90 else ''}\n")

[한국어] 원본 상위어: 보도·콘서트·2026년·공연·데뷔·2025년·일본·매체
   예시(이영지): 2026년 서울 올림픽공원 올림픽홀에서 두 번째 월드투어 콘서트 <2.0>을 개최하며 투어 규모를 이어가고 있다.

[영어] 원본 상위어: OST·fan·BTS·Japan·pop·sold·million·MBC
   예시(이영지): 월드투어 공식 굿즈(MD)를 현장 티켓 확인 절차까지 두고 사전 판매할 만큼 팬 구매 수요가 크다.

[비영어(스페인어·프랑스어·포르투갈어·인도네시아어·말레이어·필리핀어·튀르키예어 등)] 원본 상위어: México·Shopee·Übermensch·República·CDMX·Cuarta·konser·2×
   예시(이영지): 2026년 월드투어 〈2.0〉 서울 공연을 3월 7~8일 양일간 올림픽공원 올림픽홀에서 개최한다

[러시아어(키릴문자)] 원본 상위어: группы·зрителей·стал·первые·миллионов·года·около·тысяч
   예시(지드래곤 (G-Dragon)): 러시아 매체 YesAsia.ru가 G-Dragon의 2025 월드투어(17개 도시 39회 공연) 종료 및 서울 앙코르 공연을 보도: "G-Dragon дал 39…

[베트남어(라틴 확장 성조 부호)] 원본 상위어: Việt·vé·bán·đến·ảnh·đồng·đầu·khán
   예시(지드래곤 (G-Dragon)): Ca sĩ khiến khán giả phát cuồng khi hỏi 'Việt Nam, do you love me?'

[일본어] 원본 상위어: 日本·公演·東京·開催·ファン·ライブ·ドーム·デビュー
   예시(지드래곤 (G-Dragon)): 후쿠오카 야후옥돔 공연에서 5만 명의 관객이 BIGBANG 컬러 응원봉(노란색·빨간색)으로 도쿄돔 전체를 가득 메웠고, 깜짝 생일 케이크 이벤트에도 5만 관객이 …

[중국어] 원본 상위어: 香港·演唱·新浪·豆瓣·台灣·超话·网易·粉絲
   예시(이영지): 이영지 새 월드투어 첫 개최지를 타

## 5. 한계

1. 원본 토크나이저 소스(`run_lda_v6_live_reference_v7.py`)는 저장소에 없다. 일본어·중국어·태국어 버킷은 분기 조건 자체를 재현한 것이고, 일반 경로 4개 버킷은 원본의 불용어·토큰 정규식이 달라 근사치다.
2. 이 노트북은 토큰 빈도(형태소 분석)를 다루지 않는다 — 10,020건 토큰 빈도 CSV는 `v7_final_10020/analysis/tokenizer/build_bullet_token_frequency_csv_v7.py`가 산출한다.